# Apport de la télématique automobile dans la modélisation actuarielle : segmentation du risque et tarification

**Projet de modélisation en assurance — Rendu 2**
**Date : Mai 2026**

---

## Résumé

Ce projet explore l'utilisation de données télématiques automobiles dans deux problématiques actuarielles distinctes mais complémentaires : la **segmentation du risque** (identification des conducteurs à fort potentiel de sinistre par classification binaire) et la **tarification** (calcul d'une prime pure technique par modélisation fréquence × sévérité). Le travail s'appuie sur le dataset synthétique télématique de So, Boucher & Valdez (2021), qui émule un portefeuille canadien d'assurance "usage-based" avec 100 000 polices.

Trois résultats principaux émergent. Premièrement, l'apport informationnel des variables télématiques est substantiel : passer d'un modèle traditionnel (variables de souscription) à un modèle incluant les variables télématiques fait gagner **+0,11 d'AUC** en régression logistique (de 0,68 à 0,79). Deuxièmement, l'utilisation d'un modèle non-linéaire (XGBoost) apporte un gain supplémentaire de **+0,08 d'AUC** sur les mêmes variables, démontrant que les relations entre comportement de conduite et sinistralité sont fondamentalement non-linéaires. Troisièmement, un plan d'expérience factoriel 2×3 montre que pour les modèles non-linéaires, la pondération des coûts (cost-sensitive learning) doit être préférée à SMOTE pour gérer le déséquilibre des classes — un résultat empirique qui complète la littérature sur les approches algorithmiques face au déséquilibre.

Sur le volet tarification, un pipeline actuariel classique Poisson × Gamma a été ajusté en parallèle. Le pipeline produit une prime pure équilibrée à 1,9% près sur le test set (total prédit 2 157 529 vs observé 2 199 094), avec une discrimination de 9× entre le décile à plus faible risque et le décile à plus fort risque. Le modèle Poisson ne rivalise pas avec XGBoost en termes de pure capacité de ranking (AUC 0,75 vs 0,89), mais il offre des estimations directement interprétables et conformes aux exigences réglementaires actuarielles.

L'analyse complète révèle aussi un trade-off méthodologique important : les modèles optimisés pour le ranking (XGBoost + cost-weighting) souffrent d'une calibration dégradée des probabilités, problème résolu par une étape de calibration post-hoc (Platt scaling) qui restaure une calibration excellente sans coût en termes d'AUC.

**Mots-clés** : télématique, usage-based insurance, classification déséquilibrée, GLM Poisson, GLM Gamma, calibration, cost-sensitive learning, SMOTE.

---

## 1. Introduction

### 1.1 L'accessibilité des données comme moteur de transformation

L'industrie de l'assurance a toujours reposé sur l'asymétrie d'information. L'assureur fixe un prix sur la base d'un nombre limité de signaux — l'âge, le genre, l'ancienneté du permis, la zone géographique, la cylindrée du véhicule — et accepte ensuite un risque dont il ne peut connaître les détails comportementaux. Cette asymétrie produit deux effets connus depuis Akerlof (1970) : l'**anti-sélection** (les conducteurs imprudents sont attirés par les tarifs moyens) et l'**aléa moral** (le conducteur, une fois assuré, n'a plus d'incitation économique directe à conduire prudemment).

Le tarif moyen ainsi formé est donc un compromis : il sur-tarife les bons conducteurs et sous-tarife les mauvais. Réduire cette asymétrie suppose, par construction, **d'augmenter la quantité d'information disponible sur le risque individuel**. C'est précisément ce que permet aujourd'hui la révolution de la donnée.

Il y a quinze ans, observer le comportement de conduite d'un assuré relevait de la science-fiction industrielle. Pour qu'un assureur puisse savoir à quelle vitesse vous avez freiné le mardi 17 mars à 18h42, il aurait fallu installer un dispositif coûteux dans chaque véhicule, gérer la transmission des données par GPRS, stocker des téraoctets de logs sur des serveurs propriétaires. Le coût total d'un tel programme était prohibitif, et la valeur statistique des données collectées trop incertaine pour qu'un investissement massif soit justifié.

Aujourd'hui, le contexte est radicalement différent. Trois transformations technologiques convergent :

1. **Les véhicules sont eux-mêmes devenus des collecteurs de données.** Les véhicules récents intègrent en standard des centrales de traitement embarquées (ECU, OBD-II) qui enregistrent en continu vitesse, accélération, freinage, kilométrage, comportement aux carrefours. Les constructeurs eux-mêmes — Tesla, Ford, Toyota, Stellantis — opèrent des plateformes télématiques qui collectent ces données pour leurs propres usages (maintenance prédictive, mise à jour OTA, télémétrie de flotte).

2. **Le smartphone a démocratisé la captation comportementale.** Une simple application mobile, exploitant les capteurs natifs (GPS, accéléromètre, gyroscope) du smartphone du conducteur, peut produire un signal télématique d'une qualité comparable à celle d'un boîtier dédié. Le coût d'acquisition par police est quasi-nul, ce qui change l'arithmétique économique de l'usage-based insurance.

3. **Un marché de la donnée s'est structuré.** Les constructeurs, les fournisseurs de cartographie (HERE, TomTom), les opérateurs télécoms et les agrégateurs spécialisés (Cambridge Mobile Telematics, Octo Telematics) commercialisent désormais des datasets télématiques anonymisés sur des bases contractuelles. Un assureur qui ne peut pas — ou ne veut pas — déployer son propre programme UBI peut acheter de la donnée comportementale chez un tiers.

Le résultat de cette triple transformation : la **donnée télématique est passée du statut de ressource rare et coûteuse à celui de commodité accessible**. C'est ce changement structurel qui motive le présent projet.

### 1.2 Deux objectifs actuariels parallèles

L'utilisation actuarielle de la donnée télématique recouvre deux questions techniques distinctes, qui correspondent à deux moments différents du cycle de vie d'une police.

**Question 1 — Segmentation : "qui sont les conducteurs risqués ?"**

À la souscription, ou en cours de contrat, l'assureur souhaite identifier les conducteurs dont le profil suggère un risque élevé. Cette information sert à plusieurs usages opérationnels : refus de souscription pour les profils inacceptables, surveillance accrue pour les profils douteux, propositions de programmes de coaching pour les profils intermédiaires, et bonus pour les profils sains. La sortie attendue du modèle est une **décision binaire** ou un **score de risque** : "ce conducteur est-il à forte probabilité de sinistre, oui ou non ?"

Ce premier objectif relève méthodologiquement du **problème de classification binaire déséquilibrée**, le déséquilibre venant du fait que la sinistralité est par nature un événement rare (de l'ordre de 4-5 % des polices dans un portefeuille classique). Les outils adaptés sont les modèles de machine learning standards (régression logistique, forêts aléatoires, gradient boosting), avec des techniques spécifiques pour gérer le déséquilibre des classes (resampling, cost-sensitive learning).

**Question 2 — Tarification : "combien doit coûter cette police ?"**

À la souscription tarifaire, l'assureur doit fixer un prix de prime qui doit, en moyenne, couvrir le coût attendu des sinistres (la "prime pure" technique) augmenté des frais de gestion et de la marge de l'assureur. La sortie attendue du modèle est un **montant monétaire**, pas une décision binaire. Ce montant doit être **calibré** : un modèle qui prédit des primes pures systématiquement biaisées vers le haut conduit à des tarifs non-compétitifs, et un modèle biaisé vers le bas conduit à une insolvabilité technique.

Ce second objectif relève méthodologiquement de la **modélisation fréquence × sévérité**, approche actuarielle classique consistant à décomposer le coût total par police en deux facteurs : la fréquence des sinistres (combien) et la sévérité (combien chacun coûte), modélisés respectivement par une régression de Poisson et une régression Gamma.

### 1.3 Articulation du projet

Ce rapport traite ces deux questions dans un même cadre méthodologique, en utilisant le même dataset télématique. La logique de présentation suit le cycle naturel d'un projet de modélisation : exploration des données, préparation, modélisation, évaluation. À chaque étape, on traite la question 1 (classification) et la question 2 (tarification) séparément quand cela est pertinent, ou conjointement quand des outils communs s'appliquent.

Le rapport est organisé comme suit. La **Section 2** présente le contexte actuariel et la littérature de référence. La **Section 3** décrit le dataset et les variables disponibles. La **Section 4** documente le nettoyage et l'exploration des données. La **Section 5** présente les modèles de classification, depuis la régression logistique baseline jusqu'au modèle non-linéaire avec gestion du déséquilibre. La **Section 6** présente le pipeline actuariel Poisson × Gamma de tarification. La **Section 7** discute les résultats sur le test set et compare les deux approches. La **Section 8** conclut et identifie les pistes d'extension.

---

## 2. Contexte actuariel et littérature

### 2.1 Le tarif technique en assurance automobile

En assurance non-vie, la prime pure technique d'une police est définie par l'espérance du coût total des sinistres sur la période de couverture :

$$ \text{Prime pure} = E[S] = E[N] \times E[Y \mid N > 0] $$

où $N$ est le nombre de sinistres survenus et $Y$ leur montant moyen. Cette décomposition fréquence × sévérité est au cœur de la pratique actuarielle et donne lieu au pipeline classique de tarification :

1. **Modèle de fréquence** : régression de Poisson (ou variantes : binomiale négative, zero-inflated Poisson) sur le nombre de sinistres, avec l'exposition (durée de couverture) comme offset.

2. **Modèle de sévérité** : régression Gamma (ou log-normale) sur le coût moyen par sinistre, ajustée sur le sous-ensemble des polices ayant connu au moins un sinistre.

3. **Combinaison** : la prime pure prédite est obtenue par multiplication des deux espérances conditionnelles.

Cette approche, codifiée notamment par Denuit et al. (2007), bénéficie d'une longue tradition de validation empirique et d'une excellente interprétabilité : chaque coefficient peut être lu comme un effet relatif (odds ratio) sur la fréquence ou la sévérité, ce qui facilite à la fois l'audit interne et la justification réglementaire des tarifs.

### 2.2 L'apport des données télématiques

Trois travaux récents servent de cadre de référence pour ce projet.

**So, Boucher & Valdez (2020)** — "Cost-sensitive Multi-class AdaBoost for Understanding Driving Behavior with Telematics" — analyse un portefeuille canadien réel de 71 875 polices observées entre 2013 et 2016. Les auteurs traitent le problème comme une classification multi-classes (0 / 1 / 2+ sinistres) et introduisent l'algorithme **SAMME.C2**, version multi-classes de l'AdaBoost cost-sensitive. Leur résultat principal est que les variables télématiques dominent les variables traditionnelles en termes d'importance prédictive : sur le top 13 des features, seules deux variables traditionnelles (YRS_CLAIMS_FREE, CREDIT_SCORE) figurent. Le projet présent reprend la philosophie de cet article — comparer un modèle "traditionnel seul" à un modèle "traditionnel + télématique" pour quantifier l'apport informationnel — mais simplifie le problème en classification binaire (les classes 2+ étant trop peu peuplées dans la majorité des datasets pour permettre une modélisation fiable).

**So, Boucher & Valdez (2021)** — "Synthetic Dataset Generation of Driver Telematics" — décrit la méthode de génération du dataset utilisé dans ce projet. Les auteurs ont entraîné une chaîne de réseaux de neurones feedforward sur le dataset canadien réel, puis ont produit 100 000 observations synthétiques imitant les distributions et les corrélations du dataset original, sans permettre la ré-identification des assurés réels. Une procédure SMOTE étendue (avec une distribution d'interpolation en U) est utilisée pour générer l'espace des features. Le dataset synthétique a été ensuite validé par comparaison des modèles Poisson et Gamma fittés sur les données réelles vs synthétiques, avec un excellent accord sur les métriques agrégées.

**Stocksieker (2024)** — thèse de doctorat soutenue à Lyon 1 — explore la modélisation des valeurs rares et des datasets déséquilibrés dans des contextes actuariels. Les contributions méthodologiques (algorithmes WR/DA-WR, GOLIATH, DAVID) sortent du cadre de ce projet, mais la taxonomie introduite — déséquilibre côté variables explicatives vs côté variable cible, niveau données vs niveau algorithme — fournit un cadre conceptuel cohérent pour structurer les choix méthodologiques de ce travail.

### 2.3 Positionnement du projet

Le projet s'inscrit dans la lignée de So et al. (2020) en simplifiant la cible (binaire au lieu de multi-classes) pour gagner en lisibilité, tout en élargissant la palette d'outils étudiés. L'apport spécifique est triple :

1. **Plan d'expérience factoriel** comparant systématiquement modèles linéaires vs non-linéaires d'une part, et trois stratégies de gestion du déséquilibre (baseline, SMOTE, cost-sensitive) d'autre part.

2. **Implémentation d'une variante "U-shape SMOTE"** issue de Paper 2, comme test de robustesse face au SMOTE classique.

3. **Construction d'un pipeline actuariel classique en parallèle** (Poisson × Gamma avec décomposition deux-temps de la sévérité pour gérer les claims non-payés), permettant la comparaison directe entre approche ML "moderne" et approche actuarielle "classique".

---

## 3. Description des données

### 3.1 Source et nature

Le dataset utilisé est le fichier `telematics_syn-032021.csv` produit par So et al. (2021) et publiquement disponible sur le site de l'University of Connecticut. Il contient **100 000 polices d'assurance automobile synthétiques**, émulées d'un portefeuille canadien d'assurance usage-based observé entre 2013 et 2016. Le caractère synthétique du dataset signifie qu'aucune ré-identification d'assuré réel n'est possible, tout en préservant les structures statistiques (distributions marginales, corrélations, dépendances aux variables cible) du dataset source.

Le dataset comporte **52 variables** réparties en trois catégories :

- **Variables traditionnelles (11)** : informations disponibles à la souscription, sans télématique.
- **Variables télématiques (39)** : observations issues du dispositif embarqué, agrégées sur la période d'observation.
- **Variables cibles (2)** : le nombre de sinistres `NB_Claim` et le montant total des sinistres `AMT_Claim`.

### 3.2 Variables traditionnelles

Les onze variables traditionnelles couvrent les axes classiques de la tarification automobile :

| Variable | Description | Type |
|----------|-------------|------|
| Duration | Durée de couverture en jours | Numérique |
| Insured.age | Âge du conducteur principal | Numérique |
| Insured.sex | Genre (M / F) | Catégoriel |
| Marital | Statut matrimonial (Single / Married) | Catégoriel |
| Car.age | Âge du véhicule en années | Numérique |
| Car.use | Usage du véhicule (Pleasure / Commute / Farmer / Commercial) | Catégoriel |
| Credit.score | Score de crédit du conducteur | Numérique |
| Region | Type de zone (Rural / Urban) | Catégoriel |
| Annual.miles.drive | Kilométrage annuel déclaré (miles) | Numérique |
| Years.noclaims | Nombre d'années sans sinistre | Numérique |
| Territory | Code territorial (55 modalités) | Catégoriel encodé |

### 3.3 Variables télématiques

Les variables télématiques se décomposent en quatre sous-groupes thématiques :

**Exposition réelle (3 variables)**
- `Annual.pct.driven` : fraction annualisée du temps passé sur la route
- `Total.miles.driven` : distance totale parcourue pendant la période d'observation
- `Avgdays.week` : nombre moyen de jours d'utilisation par semaine

**Profil temporel de conduite (12 variables)**
- `Pct.drive.{mon,tue,wed,thr,fri,sat,sun}` : fraction du temps de conduite par jour de la semaine
- `Pct.drive.{2hrs,3hrs,4hrs}` : fraction des trajets dépassant 2/3/4 heures
- `Pct.drive.wkday` / `Pct.drive.wkend` : fraction de conduite en semaine vs weekend
- `Pct.drive.rusham` / `Pct.drive.rushpm` : fraction de conduite aux heures de pointe matin/soir

**Comportement de conduite (12 variables)**
- `Accel.{06,08,09,11,12,14}miles` : nombre d'accélérations brusques par 1000 miles, à différents seuils d'intensité (de 6 à 14 mph/s)
- `Brake.{06,08,09,11,12,14}miles` : nombre de freinages brusques par 1000 miles, mêmes seuils

**Comportement aux carrefours (10 variables)**
- `Left.turn.intensity{08,09,10,11,12}` : nombre de virages à gauche par 1000 miles, à différentes intensités
- `Right.turn.intensity{08,09,10,11,12}` : équivalent pour les virages à droite

### 3.4 Variables cible et structure du déséquilibre

La variable cible primaire `NB_Claim` prend les valeurs entières de 0 à 3, avec la distribution suivante dans le dataset complet :

| Valeur | Effectif | Fréquence |
|--------|----------|-----------|
| 0 sinistre | 95 728 | 95,73 % |
| 1 sinistre | 4 060 | 4,06 % |
| 2 sinistres | 199 | 0,20 % |
| 3 sinistres | 13 | 0,01 % |

Le déséquilibre est marqué : 95,73 % de polices sans sinistre contre 4,27 % avec au moins un sinistre, soit un **ratio de déséquilibre d'environ 22:1**. Les classes 2 et 3 sont si peu peuplées (199 et 13 observations) qu'aucune modélisation directe de ces classes n'est statistiquement défendable : on choisit donc de **collapser la cible en variable binaire `has_claim ∈ {0, 1}`**, où `has_claim = 1` si `NB_Claim > 0`.

La variable `AMT_Claim` (montant total des sinistres) est conservée pour le pipeline actuariel de Section 6. Elle prend des valeurs dans $[0, 138 766]$, avec un point de masse à 0 (les non-sinistrés) et une distribution fortement asymétrique pour les sinistrés (médiane ≈ 2 200 $, max ≈ 138 000 $).

Notons un point important : ~10% des polices avec `NB_Claim > 0` ont `AMT_Claim = 0`. Ces observations correspondent économiquement à des claims déclarés mais non-indemnisés (claim refusé, montant inférieur à la franchise, retrait du sinistré). Elles seront traitées spécifiquement dans la modélisation de sévérité (Section 6).

---

## 4. Préparation et exploration des données

### 4.1 Nettoyage et imputation

L'inspection des distributions marginales a révélé plusieurs incohérences ponctuelles, traitées comme suit :

- **Durations supérieures à 365 jours** (artefacts de génération synthétique) : remplacement par le mode (366 → 365).
- **Âges hors de l'intervalle [18, 75]** : remplacement par la médiane.
- **Âges de véhicule négatifs** (cas isolés) : tirage aléatoire dans la distribution empirique des âges positifs.
- **Incohérences âge/ancienneté sans sinistre** (`Years.noclaims > Age`) : capping de `Years.noclaims` à la valeur de `Age`.

L'inspection des variables télématiques n'a révélé aucune valeur aberrante nécessitant un traitement spécifique. Les fractions compositionnelles (jours de la semaine, weekend/semaine) somment exactement à 1 par construction, ce qui sera traité en Section 5.1.

### 4.2 Variables dérivées

Trois variables ont été créées pour usage ultérieur :

- `Expected_miles = Annual_miles_drive × Duration / 365` : kilométrage attendu sur la période, basé uniquement sur des informations de souscription.
- `Exposure = Duration / 365` : exposition fractionnaire en années, utilisée comme offset dans les modèles GLM.
- `mileage_ratio = Total_miles_driven / Expected_miles` : ratio entre kilométrage observé et kilométrage déclaré, indicateur de la fiabilité de la déclaration initiale.

Une valeur infinie a été corrigée pour 3 polices ayant `Annual_miles_drive = 0` (remplacement par la médiane du `mileage_ratio`).

### 4.3 Analyse exploratoire — variables traditionnelles

L'analyse univariée révèle des relations modérées mais cohérentes entre les variables traditionnelles et la sinistralité. Les résultats clés :

- **Âge** : la sinistralité présente un profil en U classique, avec des taux légèrement plus élevés chez les conducteurs très jeunes (< 25 ans) et très âgés (> 75 ans).
- **Score de crédit** : relation négative monotone — les conducteurs aux scores élevés ont une sinistralité plus faible. Cette relation, bien établie dans la littérature actuarielle nord-américaine, est l'un des facteurs traditionnels les plus prédictifs sur ce dataset.
- **Âge du véhicule** : relation négative également — les véhicules plus anciens sont associés à une sinistralité plus faible, probablement parce qu'ils sont conduits avec plus de précaution et sur de plus courtes distances.
- **Genre, statut matrimonial, territoire** : effets très faibles et statistiquement non-significatifs (cf. Section 5.1).

La matrice de corrélation entre les variables traditionnelles et `NB_Claim` montre des corrélations généralement faibles (en valeur absolue inférieures à 0,06), ce qui est attendu pour un problème de classification déséquilibrée — la corrélation linéaire est une mesure peu adaptée à ce type de relation.

### 4.4 Analyse exploratoire — variables télématiques

L'analyse univariée des variables télématiques révèle des relations beaucoup plus marquées avec la sinistralité, en particulier pour les variables d'exposition réelle :

- **Total_miles_driven** : relation positive forte. Les drivers avec une exposition supérieure à 15 000 miles ont un taux de sinistre approchant les 10 %, contre moins de 2 % pour les drivers avec moins de 5 000 miles.
- **Annual_pct_driven** : relation positive également. Plus le véhicule est utilisé sur la période, plus la probabilité de sinistre est élevée — ce qui est intuitivement attendu.
- **Brake.xxmiles** : relations positives à toutes les intensités. Les freinages brusques fréquents sont associés à une augmentation marquée de la sinistralité, particulièrement aux seuils intermédiaires (08, 09, 11 mph/s).
- **Accel.xxmiles** : relations positives mais moins marquées que pour les freinages. Cela suggère que les **freinages brusques sont un meilleur indicateur de risque** que les accélérations brusques, ce qui est cohérent avec l'intuition que les freinages d'urgence reflètent des situations de conduite réellement dangereuses.

L'exploration par bins de variables continues confirme la **non-linéarité de plusieurs relations**, notamment :

- L'âge (relation en U)
- Le score de crédit (effet plus marqué aux extrêmes)
- Les variables de comportement (effets de seuil — un freinage brusque occasionnel a peu d'effet, mais à partir d'un certain seuil de fréquence, l'effet devient significatif)

Cette observation motive le choix d'inclure des modèles non-linéaires (XGBoost) en complément de la régression logistique.

---

## 5. Modèles de classification — segmentation du risque

### 5.1 Préparation des données

**Cible binaire et splits.** La variable cible `has_claim = 1[NB_Claim > 0]` est créée. Le dataset est divisé en trois ensembles selon une stratification sur la cible : 70 % train (69 999 obs, 2 990 positifs), 15 % validation (15 001 obs, 641 positifs), 15 % test (15 000 obs, 641 positifs). La stratification préserve le ratio 4,27 % de positifs dans chacun des trois sous-ensembles.

**Sélection des features.** On distingue deux ensembles de variables explicatives :

- **Ensemble traditionnel (11 variables)** : `Duration, Age, Gender, Car_age, Marital, Car_use, Credit_score, Region, Annual_miles_drive, Years_noclaims, Territory`.
- **Ensemble complet (48 variables)** : ensemble traditionnel + 37 variables télématiques (après retrait de `Pct_drive_sun` et `Pct_drive_wkend` pour éviter la colinéarité parfaite due aux contraintes compositionnelles).

Les variables dérivées `Expected_miles` et `Exposure` ont initialement été incluses dans l'ensemble traditionnel mais ont été retirées après diagnostic de colinéarité parfaite avec `Duration` (cf. Section 5.2). La variable `mileage_ratio` reste dans l'ensemble télématique car elle dépend d'une observation télématique (`Total_miles_driven`).

**Préprocesseur.** Un pipeline `ColumnTransformer` applique un `StandardScaler` aux variables numériques et un `OneHotEncoder(drop='first')` aux variables catégorielles. Le pipeline est ajusté **uniquement sur les données d'entraînement** puis appliqué aux ensembles de validation et de test, pour éviter toute fuite d'information.

### 5.2 Diagnostic de colinéarité

Une régression logistique a été ajustée avec `statsmodels` sur l'ensemble traditionnel initial (13 variables), pour disposer d'une table de coefficients avec p-values. La sortie a révélé un problème :

```
Duration         coef = 0.353   std_err = NaN   z = NaN   p = NaN
Exposure         coef = 0.353   std_err = NaN   z = NaN   p = NaN
```

Les variables `Duration` et `Exposure` se voyaient attribuer le même coefficient, et `statsmodels` était incapable de calculer leur erreur standard. La cause est immédiate : `Exposure = Duration / 365`, donc après standardisation les deux variables sont **mathématiquement identiques** — la matrice de design est rang-déficiente.

De même, les coefficients de `Annual_miles_drive` (+0,580) et `Expected_miles` (-0,529) étaient quasi-opposés, signe que le modèle utilisait essentiellement leur différence comme prédicteur effectif — phénomène classique de colinéarité forte (`Expected_miles ≈ Annual_miles_drive × Duration/365`, et `Duration` varie peu).

**Décision : retrait de `Exposure` et `Expected_miles` de l'ensemble traditionnel.** L'ensemble traditionnel passe de 13 à 11 variables. Le ré-ajustement produit une table de coefficients propre, sans NaN, avec des effets directement interprétables. L'AUC perd moins de 0,002 — le signal était entièrement redondant.

### 5.3 Modèle 1 — Régression logistique sur variables traditionnelles

Le premier modèle de référence est une régression logistique sur les 11 variables traditionnelles uniquement. Cette baseline répond à la question : **quelle performance prédictive obtient-on sans aucune information télématique ?**

L'ajustement avec `sklearn.LogisticRegression` (régularisation L2 par défaut) donne sur la validation :
- **AUC-ROC = 0,6779**
- Au seuil 0,5 : recall sur les positifs = 0 % (le modèle prédit "pas de sinistre" pour tout le monde)
- Au seuil G-mean optimal (0,0440) : recall(1) = 60,8 %, recall(0) = 64,1 %, **G-mean = 0,625**

Le tableau des coefficients (avec `statsmodels`) confirme la hiérarchie attendue :

| Variable | Coefficient | z | p-value | Effet (OR) |
|----------|-------------|---|---------|------------|
| Duration | +0,430 | 15,1 | < 10⁻⁵¹ | 1,538 |
| Car_age | -0,311 | -15,0 | < 10⁻⁵⁰ | 0,732 |
| Credit_score | -0,239 | -13,6 | < 10⁻⁴² | 0,787 |
| Annual_miles_drive | +0,110 | 5,8 | < 10⁻⁸ | 1,116 |
| Region_Urban | +0,195 | 3,8 | < 10⁻⁴ | 1,215 |
| Years_noclaims | -0,123 | -3,8 | < 10⁻⁴ | 0,884 |
| Car_use_Commute | -0,197 | -2,0 | 0,043 | 0,821 |
| Marital_Single | +0,054 | 1,3 | 0,197 | 1,056 |
| Age | +0,024 | 0,7 | 0,466 | 1,025 |
| Territory | -0,006 | -0,3 | 0,747 | 0,994 |
| Gender_Male | +0,006 | 0,2 | 0,867 | 1,006 |

Lecture actuarielle : six variables sont fortement significatives (p < 0,001), `Car_use` est marginalement significatif. **Quatre variables — `Age`, `Gender`, `Marital`, `Territory` — ne le sont pas du tout** (p > 0,2). C'est un constat actuariellement notable : les variables démographiques classiques de tarification ne portent **aucun signal prédictif** sur ce dataset, ce qui contraste avec les pratiques tarifaires traditionnelles.

Un test du rapport de vraisemblance comparant le modèle complet à un modèle réduit aux seules variables significatives confirme ce résultat : χ² = 2,14, ddl = 4, **p = 0,71** — on ne peut pas rejeter l'hypothèse que les coefficients de ces quatre variables sont conjointement nuls. L'AIC favorise le modèle réduit (Δ = -5,86).

### 5.4 Modèle 2 — Régression logistique sur variables traditionnelles + télématiques

Le deuxième modèle ajoute les 37 variables télématiques au modèle précédent. La performance sur la validation est :
- **AUC-ROC = 0,7873** (gain de **+0,1110** par rapport au modèle traditionnel)
- Au seuil G-mean optimal (0,0440) : recall(1) = 72,2 %, recall(0) = 72,4 %, **G-mean = 0,723**

Le gain de 11 points d'AUC mesure directement **l'apport informationnel des variables télématiques au-delà des variables traditionnelles**. Cet apport est massif et confirme l'hypothèse centrale du projet : les données télématiques enrichissent significativement la prédictivité d'un modèle de sinistralité.

### 5.5 Modèle 3 — XGBoost sur variables traditionnelles + télématiques

Pour quantifier l'apport spécifique de la modélisation non-linéaire, on ajuste un troisième modèle — XGBoost — sur le même ensemble complet de variables. Les hyperparamètres choisis (sans tuning fin) sont : 1000 arbres avec early stopping (`early_stopping_rounds=30`), profondeur maximale 4, learning rate 0,05, échantillonnage stochastique (subsample = colsample = 0,8).

Sur la validation, XGBoost s'arrête à l'itération 495 et obtient :
- **AUC-ROC = 0,8878** (gain de **+0,0833** par rapport à la régression logistique sur les mêmes variables)
- Au seuil G-mean optimal (0,0479) : recall(1) = 81,9 %, recall(0) = 79,4 %, **G-mean = 0,807**

Le gain par rapport à la régression logistique mesure **l'apport spécifique des non-linéarités et des interactions** que XGBoost capture mais que le GLM ne représente pas. Combiné au gain de la télématique, l'écart total entre la baseline traditionnelle (LR) et le meilleur modèle (XGBoost full) est de **+0,194 d'AUC**, soit une amélioration relative de 29 %.

**Importance des variables.** L'analyse de feature importance de XGBoost (basée sur le gain) place les variables suivantes dans le top 10 :

1. `Total_miles_driven` (télématique) — 0,077
2. `Brake_11miles` (télématique) — 0,067
3. `Brake_08miles` (télématique) — 0,043
4. `Annual_pct_driven` (télématique) — 0,042
5. `Exposure` (traditionnel — note : retiré ensuite, voir Section 5.2)
6. `Duration` (traditionnel) — 0,030
7. `Brake_09miles` (télématique) — 0,029
8. `Car_age` (traditionnel) — 0,024
9. `Credit_score` (traditionnel) — 0,024
10. `Marital_Single` (traditionnel) — 0,021

Quatorze des vingt premières features sont télématiques. Les deux variables les plus importantes — `Total_miles_driven` et `Brake_11miles` — sont télématiques et reflètent respectivement l'**exposition réelle** (combien le conducteur conduit effectivement) et un **comportement à risque** (freinages brusques à intensité élevée). Ce résultat est cohérent avec So et al. (2020) qui placent également les variables d'exposition et de freinage en tête.

### 5.6 Phase comparative : traitement du déséquilibre

Au seuil 0,5, tous les modèles précédents souffrent du déséquilibre 22:1 : ils prédisent "pas de sinistre" pour la quasi-totalité des observations. Au seuil G-mean optimal, ils deviennent opérationnels mais nécessitent un tuning de seuil post-hoc. Pour intégrer le traitement du déséquilibre dans la phase de modélisation elle-même, on a comparé systématiquement trois stratégies, dans un **plan d'expérience factoriel 2 × 3** : modèles linéaire (LR) vs non-linéaire (XGBoost), chacun avec trois traitements (baseline, SMOTE, cost-weighted).

**SMOTE-NC personnalisé.** Le dataset comprenant à la fois des variables numériques et catégorielles, on a implémenté manuellement une version SMOTE-NC qui interpole linéairement les variables numériques entre une observation minoritaire et un de ses plus proches voisins, et copie les valeurs catégorielles de l'observation source. Une variante "U-shape" (Beta(0.5, 0.5) au lieu de Uniform(0,1) pour le coefficient d'interpolation), inspirée de Paper 2, a été testée en parallèle.

**Cost-sensitive learning.** Pour la régression logistique, on a utilisé `class_weight='balanced'`, qui applique un poids inversement proportionnel à la fréquence de chaque classe. Pour XGBoost, on a utilisé `scale_pos_weight = n_neg / n_pos ≈ 22,4`, qui multiplie la contribution des positifs au gradient.

**Résultats sur la validation** (au seuil G-mean optimal) :

| Modèle | AUC | G-mean (opt) | G-mean (thr=0.5) |
|--------|-----|--------------|------------------|
| LR baseline | 0,7873 | 0,723 | 0,000 |
| LR + SMOTE uniforme | 0,7890 | 0,721 | 0,720 |
| LR + SMOTE U-shape | 0,7890 | 0,721 | 0,720 |
| LR + cost-weighted | 0,7905 | 0,725 | 0,722 |
| XGBoost baseline | 0,8878 | 0,807 | 0,308 |
| XGBoost + SMOTE uniforme | 0,8528 | 0,775 | 0,416 |
| XGBoost + SMOTE U-shape | 0,8577 | 0,778 | 0,441 |
| **XGBoost + cost-weighted** | **0,8883** | **0,813** | **0,803** |

Trois conclusions ressortent clairement de ce tableau :

1. **Pour la régression logistique, les trois traitements sont équivalents en AUC** (0,789–0,791, écart < 0,002). L'apport principal n'est pas le ranking mais l'opérationnalité au seuil 0,5 — SMOTE et cost-weighted permettent au modèle d'être déployable directement, sans tuning de seuil.

2. **Pour XGBoost, SMOTE dégrade significativement l'AUC** (-0,035 pour SMOTE uniforme). Ce résultat empirique confirme une recommandation théorique souvent négligée : les modèles à base d'arbres gèrent le déséquilibre nativement par leurs critères de split (gain d'information, gini), qui ne dépendent pas linéairement des proportions de classes. Ajouter des observations synthétiques par interpolation produit du bruit que les arbres peuvent sur-apprendre.

3. **Le cost-weighted XGBoost est le grand vainqueur** : il maintient l'AUC du baseline (0,8883 vs 0,8878) tout en rendant le modèle opérationnel au seuil 0,5 (G-mean = 0,803 vs 0,308). Aucun trade-off — c'est un gain pur.

**La variante U-shape SMOTE n'apporte rien sur ce dataset** (différence d'AUC < 0,005 vs SMOTE uniforme, à la fois pour LR et XGBoost). La taille de la classe minoritaire (~3000 observations dans le train) rend probablement la distribution d'interpolation peu critique. Ce résultat négatif est cohérent avec l'intuition de Paper 2 selon laquelle U-shape SMOTE serait surtout utile pour des classes minoritaires très petites (< 500 obs) où la préservation des cas extrêmes est critique.

### 5.7 Évaluation finale sur le test set

Quatre modèles candidats issus des phases précédentes sont évalués sur le **test set** (15 000 obs, 641 positifs), qui n'a été utilisé à aucun moment des phases de sélection. Le seuil de décision pour les métriques opérationnelles est le **seuil G-mean optimal mesuré sur la validation** (jamais sur le test), pour éviter toute fuite.

Les résultats sont accompagnés d'**intervalles de confiance bootstrap à 95 %**, calculés par 1000 rééchantillonnages du test set avec remise.

| Modèle | AUC test [CI 95%] | G-mean [CI 95%] | Recall(1) [CI 95%] |
|--------|-------------------|------------------|---------------------|
| LR baseline | 0,778 [0,761 ; 0,797] | 0,709 [0,692 ; 0,729] | 0,697 [0,664 ; 0,734] |
| LR + cost-weighted | 0,782 [0,765 ; 0,800] | 0,707 [0,690 ; 0,725] | 0,697 [0,663 ; 0,733] |
| XGBoost baseline | 0,891 [0,878 ; 0,904] | 0,804 [0,789 ; 0,819] | 0,811 [0,782 ; 0,842] |
| **XGBoost + cost-weighted** | **0,889 [0,876 ; 0,901]** | **0,808 [0,792 ; 0,822]** | **0,816 [0,785 ; 0,847]** |

**Comparaison validation vs test.** Les écarts d'AUC entre validation et test sont tous inférieurs à 0,01, ce qui indique une excellente généralisation : les conclusions de la phase 5.6 tiennent sur le test set sans surapprentissage.

**Verdict.** Les CIs des modèles XGBoost ne se chevauchent pas avec celles des modèles LR (0,876 minimum vs 0,800 maximum) — l'avantage de XGBoost est statistiquement significatif. Entre les deux variants XGBoost, les CIs se chevauchent (0,876–0,901 vs 0,876–0,904) : les deux modèles sont statistiquement équivalents en AUC, avec un avantage marginal pour le cost-weighted en G-mean opérationnel.

### 5.8 Calibration des probabilités

L'AUC mesure la qualité de **ranking** : un AUC élevé garantit que le modèle ordonne correctement les drivers du moins risqué au plus risqué. Mais l'AUC ne dit rien sur la **valeur numérique** des probabilités prédites.

Pour le cas d'usage **classification / segmentation** (Section 1.2 — question 1), seul le ranking importe : on prend les K conducteurs les plus risqués pour les soumettre à une procédure spécifique, l'ordre exact des probabilités n'a pas besoin d'être étalonné. Mais pour la **tarification** (question 2), c'est précisément la valeur de la probabilité qui détermine la prime : un modèle qui prédit `P = 0,30` quand le vrai risque est `P = 0,04` produira des primes 7× trop élevées.

L'analyse de calibration sur le test set révèle un trade-off important :

| Modèle | Brier | ECE | Mean(prob prédite) | Taux réel |
|--------|-------|-----|---------------------|-----------|
| XGBoost baseline | 0,0310 | 0,009 | 0,039 | 0,043 |
| XGBoost + cost-weighted | 0,0904 | 0,169 | 0,212 | 0,043 |

**XGBoost baseline est exceptionnellement bien calibré** : ECE = 0,009 (moins de 1 point de pourcentage d'erreur moyenne), mean(prob) = 0,039 quasi-identique au taux réel 0,043. Les probabilités prédites sont directement utilisables en tarification.

**XGBoost + cost-weighted, en revanche, est massivement miscalibré** : la probabilité moyenne prédite est de 21 %, près de 5 fois le taux réel. Ce biais est attendu théoriquement : le cost-weighting "fait croire" au modèle que la classe positive est plus fréquente qu'elle ne l'est, ce qui inflate les probabilités prédites.

**Résolution par calibration post-hoc.** On peut corriger ce biais sans perte d'AUC en appliquant une transformation monotone aux probabilités prédites, ajustée sur la validation :

| Variant | Brier | ECE | AUC | Mean(prob) |
|---------|-------|-----|-----|------------|
| Cost-weighted brut | 0,143 | 0,259 | 0,849 | 0,302 |
| + Platt scaling | 0,036 | 0,004 | 0,849 | 0,042 |
| + Isotonic regression | 0,035 | 0,003 | 0,847 | 0,042 |

Les deux méthodes (Platt scaling, régression isotonique) restaurent une calibration excellente — Brier divisé par 4, ECE divisé par 65 — sans coût en AUC pour Platt et avec un coût négligeable (-0,003) pour Isotonic. La probabilité moyenne prédite passe de 30 % à 4,2 %, en accord avec le taux réel.

**Synthèse.** Pour un déploiement opérationnel :

- En **segmentation pure**, XGBoost + cost-weighted suffit. Pas besoin de calibration.
- En **tarification**, deux options équivalentes : (a) XGBoost baseline (déjà calibré naturellement), (b) XGBoost + cost-weighted + Platt scaling.

---

## 6. Pipeline actuariel — tarification fréquence × sévérité

### 6.1 Motivation

La Section 5 a traité la **classification binaire** (`has_claim ∈ {0,1}`). Cette approche répond à la question 1 de l'introduction : quels drivers sont les plus risqués ? Mais elle ne répond pas à la question 2 : combien doit coûter chaque police ?

Pour produire une **prime pure technique**, il faut un modèle qui intègre :

1. La **fréquence** des sinistres : non plus une probabilité binaire mais un nombre attendu $E[N]$ qui dépend de l'exposition (durée de couverture).
2. La **sévérité** des sinistres : le montant moyen attendu par sinistre $E[Y \mid N>0]$.
3. La **prime pure** = $E[N] \times E[Y \mid N>0]$.

Cette section construit ce pipeline en parallèle des modèles de classification, en utilisant la **même structure de splits** (70/15/15 stratifié) pour permettre la comparaison directe.

### 6.2 Modèle de fréquence — régression de Poisson

La régression de Poisson modélise le nombre de sinistres comme :

$$ \log(\lambda_i) = X_i \beta + \log(\text{Exposure}_i) $$

où $\lambda_i$ est le nombre attendu de sinistres et $\text{Exposure}_i = \text{Duration}_i / 365$ joue le rôle d'**offset** (variable d'exposition ne nécessitant pas d'estimation de coefficient). Sklearn implémente l'offset via le paramètre `sample_weight` du `PoissonRegressor`.

Le modèle est ajusté sur le train avec une régularisation L2 modérée (`alpha = 1.0`) pour stabiliser l'estimation.

**Résultats sur le test set** :
- Σ prédite = 678,8 sinistres ; Σ observée = 679 sinistres
- Mean(prédite) = 0,0453 ; Mean(observée) = 0,0453
- AUC pour ranker `has_claim` : 0,755

Le modèle Poisson est **excellemment calibré au niveau agrégé** (Σ prédit ≈ Σ observé à l'unité près), ce qui est la propriété centrale recherchée pour un modèle technique. Sa capacité de discrimination (AUC 0,75) est inférieure à celle de XGBoost (0,89) mais supérieure à celle de LR-baseline (0,78), ce qui est cohérent : Poisson est un modèle linéaire (généralisé), au même titre que LR, mais optimise une log-vraisemblance de comptage plutôt qu'une cross-entropie binaire.

### 6.3 Modèle de sévérité — décomposition en deux étapes

**Problème de support.** La régression Gamma exige des cibles strictement positives. Or, ~10 % des polices avec `NB_Claim > 0` ont `AMT_Claim = 0` (303 obs sur 2 990 dans le train). Ces observations sont économiquement légitimes — un sinistre déclaré peut être refusé, couvert par la franchise, ou retiré par l'assuré — mais incompatibles avec le support strictement positif de la distribution Gamma.

La solution actuarielle standard est de **décomposer la sévérité en deux étapes** :

$$ E[Y \mid N>0] = \underbrace{P(Y>0 \mid N>0)}_{\text{Étape 1 : logistique}} \times \underbrace{E[Y \mid N>0, Y>0]}_{\text{Étape 2 : Gamma}} $$

**Étape 1.** Régression logistique sur le sous-ensemble des claimants ($N > 0$), prédisant la probabilité que le claim soit indemnisé ($Y > 0$). Sur le test, le modèle prédit en moyenne 90,4 % de probabilité de paiement, contre 89,5 % observés — bonne calibration.

**Étape 2.** Régression Gamma sur le sous-ensemble des claimants payés ($N > 0$ et $Y > 0$, soit 2 687 obs en train), avec :
- Cible : coût moyen par sinistre `AMT_Claim / NB_Claim`
- Pondération : par `NB_Claim` (un driver avec 2 sinistres compte pour deux observations)
- Régularisation : `alpha = 1.0` (forte L2, nécessaire pour éviter les extrapolations extrêmes — sans régularisation, le modèle prédit jusqu'à 10¹⁷ sur certaines observations test, artéfact classique du GLM Gamma quand le ratio claimants/features est faible)

Sur le test, la sévérité prédite moyenne (parmi les claimants payés) est de 3 412, contre 3 576 observée. La sévérité combinée $E[Y \mid N>0]$ est de 3 026, contre 3 234 observée — biais de 6 %, attribué à la régularisation forte qui shrinke les prédictions vers la moyenne globale.

### 6.4 Prime pure et lift table

La prime pure technique est le produit fréquence × sévérité combinée :

$$ \text{PurePremium}_i = \lambda_i \times \text{Exposure}_i \times P(Y>0 \mid N>0)_i \times E[Y \mid N>0, Y>0]_i $$

**Validation au niveau agrégé** :
- Total prédit (test) : **2 157 529**
- Total observé (test) : 2 199 094
- Ratio observé / prédit : **1,019**

Le ratio est à 1,9 % de la valeur cible (1,0), ce qui est dans la norme actuarielle d'un tarif équilibré (typiquement ±5 %). Ce résultat valide le pipeline complet — fréquence et sévérité combinées produisent une prime pure techniquement défendable.

**Validation au niveau de la segmentation** : on trie les 15 000 polices test par prime pure prédite et on les divise en 10 groupes de 1 500 polices (déciles).

| Décile | N | Prime prédite | Coût observé | Ratio obs/pred | Taux sinistre | Lift |
|--------|---|--------------|--------------|----------------|----------------|------|
| 1 (faible) | 1500 | 32,4 | 6,8 | 0,21 | 1,0 % | 0,23× |
| 2 | 1500 | 44,4 | 7,7 | 0,17 | 1,0 % | 0,23× |
| 3 | 1500 | 65,6 | 70,6 | 1,08 | 3,0 % | 0,70× |
| 5 | 1500 | 107,2 | 73,6 | 0,69 | 4,0 % | 0,94× |
| 7 | 1500 | 135,4 | 152,6 | 1,13 | 5,0 % | 1,17× |
| 10 (élevé) | 1500 | 514,7 | 510,3 | **0,99** | 9,0 % | **2,11×** |

Trois lectures importantes :

1. **Discrimination forte au niveau agrégé** : le décile 10 a une prime prédite (et un coût observé) **15 fois supérieure** à celle du décile 1, et un taux de sinistre **9 fois supérieur**.

2. **Calibration excellente au sommet** : pour le décile 10, le ratio observé/prédit est de 0,99 — le modèle est presque parfaitement étalonné pour les polices à haut risque, qui sont précisément celles où la précision du tarif compte le plus.

3. **Sur-tarification modérée des polices à faible risque** : pour les déciles 1 et 2, la prime prédite est ~5× plus élevée que le coût observé. Ce biais est une conséquence directe de la régularisation Gamma forte, qui shrinke les prédictions vers la moyenne globale. Pour un déploiement opérationnel, on chercherait à optimiser la régularisation par cross-validation pour rééquilibrer ce biais ; pour ce projet académique, on l'accepte comme limitation documentée.

### 6.5 Comparaison ML vs actuariel

Le tableau ci-dessous résume les performances de toutes les approches étudiées sur le test set, pour la même tâche de ranking de `has_claim` :

| Modèle | AUC test | Approche | Sortie native |
|--------|----------|----------|---------------|
| Poisson rate × exposure | 0,755 | GLM linéaire | E[N] |
| LR baseline | 0,778 | GLM linéaire | P(claim) |
| XGBoost baseline | 0,891 | Arbres non-linéaires | P(claim) |
| XGBoost + cost-weighted | 0,889 | Arbres + pondération | P(claim) |

**Lecture.** XGBoost reste le meilleur ranker (par 11 points d'AUC sur Poisson). C'est attendu : il capture des non-linéarités et des interactions que les modèles linéaires ne représentent pas. **Mais Poisson reste l'unique modèle parmi ces quatre qui produit nativement une estimation cohérente de E[N]**, donc une prime pure technique. Les modèles binaires (LR, XGBoost) prédisent une probabilité de sinistre, pas un nombre attendu — la conversion vers une prime pure exige une modélisation séparée de la sévérité, et une calibration soignée des probabilités.

Les deux approches répondent donc à des questions complémentaires :

| Question | Modèle adapté |
|----------|---------------|
| Qui sont les drivers à fort risque ? (segmentation) | XGBoost + cost-weighted |
| Combien doit coûter chaque police ? (tarification) | Poisson × Gamma (deux étapes) |

Pour un déploiement réel, l'**approche hybride** semble optimale : utiliser XGBoost pour la segmentation et le routage opérationnel (refus, surveillance), et utiliser le pipeline actuariel Poisson × Gamma pour le calcul du tarif technique.

---

## 7. Discussion

### 7.1 L'apport mesurable de la télématique

Le résultat principal du projet, mesuré sur le test set, est un **gain d'AUC de +0,194** entre la baseline traditionnelle (régression logistique sur 11 variables de souscription) et le meilleur modèle (XGBoost sur 48 variables). Ce gain se décompose proprement :

- **+0,113 d'AUC** lié à l'**ajout des variables télématiques** au modèle linéaire (passage de LR-trad à LR-full)
- **+0,083 d'AUC** lié à l'**utilisation d'un modèle non-linéaire** sur les mêmes variables (passage de LR-full à XGBoost)

Cette décomposition est importante car elle montre que les deux gains sont **additifs** et de magnitude comparable. La donnée télématique apporte un signal informationnel substantiel, et la capture des interactions non-linéaires entre ces variables apporte un gain comparable. Aucun de ces deux gains ne suffit individuellement à atteindre la performance finale ; les deux sont nécessaires.

Pour l'industrie, ce résultat suggère qu'investir dans **la donnée seule** (mise en place d'un programme UBI) sans investir dans **les modèles non-linéaires** pour l'exploiter laisse une part substantielle du gain potentiel sur la table. Inversement, des modèles non-linéaires sophistiqués sur des données traditionnelles seules atteignent un plateau de performance modeste (~0,68 d'AUC dans notre cas).

### 7.2 Le trade-off ranking vs calibration

Une découverte méthodologique importante du projet est l'opposition entre **qualité de ranking** (AUC, G-mean) et **qualité de calibration** (Brier, ECE) selon le traitement du déséquilibre :

- Le **cost-sensitive learning** améliore le ranking opérationnel et rend le modèle utilisable au seuil 0,5, mais détériore massivement la calibration des probabilités (mean prédite = 21 % vs taux réel 4,3 %).
- Le **modèle baseline** (sans traitement spécifique) est moins opérationnel au seuil 0,5 mais ses probabilités sont directement utilisables pour la tarification (mean prédite = 3,9 % vs taux réel 4,3 %).

Cette tension reflète le fait que **"meilleur classificateur" et "meilleur estimateur de probabilités" sont deux objectifs différents**. Le projet montre qu'on peut avoir les deux simultanément en utilisant le cost-weighting pour le ranking et la **calibration post-hoc** (Platt scaling) pour restaurer les probabilités. Cette combinaison — XGBoost + cost-weighting + Platt — est le pipeline le plus opérationnel pour un usage industriel mixte (segmentation et tarification).

### 7.3 La leçon sur SMOTE et les modèles à base d'arbres

Le plan d'expérience factoriel a produit un **résultat empirique négatif intéressant** : pour XGBoost, SMOTE dégrade l'AUC de 3,5 points (0,888 → 0,853). Ce résultat ajoute de l'évidence empirique à une recommandation théorique souvent négligée par les praticiens.

Le mécanisme est compréhensible : les arbres de décision gèrent le déséquilibre nativement par leurs critères de split (gain d'information, gini), qui dépendent des proportions de classes via la pureté des nœuds, mais pas via une fonction de loss linéaire vis-à-vis de ces proportions. Ajouter des observations synthétiques par interpolation linéaire entre points minoritaires peut introduire des artéfacts dans des régions de l'espace où les arbres surapprennent ces interpolations au détriment des vraies relations.

La recommandation pratique est claire : **pour les modèles à base d'arbres, préférer le cost-sensitive learning à SMOTE**. Pour les modèles linéaires, les deux sont équivalents en performance ; le choix peut alors être fait sur des critères secondaires (simplicité d'implémentation, interprétabilité, contraintes de production).

### 7.4 Limites du travail

Plusieurs limitations méritent d'être discutées.

**Données synthétiques.** Le dataset utilisé est synthétique (généré par un réseau de neurones imitant un dataset canadien réel). Les AUC obtenus (0,68 pour LR-trad, 0,89 pour XGBoost-full) sont supérieurs aux attentes communiquées par l'encadrant pour des datasets réels (typiquement 0,55–0,62 pour LR-trad). Cette inflation est probablement due au caractère synthétique des données : le générateur neural, par construction, apprend des relations plus nettes que celles d'un dataset réel bruité. Les conclusions méthodologiques (apport relatif de la télématique, comparaison SMOTE vs cost-weighted, etc.) tiennent ; les **niveaux absolus de performance** sont à considérer comme une borne supérieure optimiste.

**Pas de validation croisée.** Le projet utilise un seul split train/validation/test avec graine fixée. Les intervalles de confiance bootstrap sur le test capturent l'incertitude due à l'échantillonnage du test, mais pas la variance due au choix du split. Une validation croisée 5-fold aurait fourni une estimation plus robuste. Ce choix a été fait pour des raisons de coût computationnel et de simplicité de présentation.

**Hyperparamètres XGBoost non tunés.** Les hyperparamètres XGBoost (1000 arbres, depth 4, lr 0,05) sont raisonnables mais n'ont pas été optimisés par recherche bayésienne ou gridsearch. Une optimisation pourrait apporter quelques points d'AUC supplémentaires, sans changer les conclusions qualitatives.

**Sur-régularisation Gamma.** Le pipeline actuariel utilise une régularisation Gamma forte (`alpha = 1.0`) qui produit une sur-tarification des déciles à faible risque. Une optimisation par cross-validation de cette régularisation est l'extension la plus directe du travail.

**U-shape SMOTE non concluant.** La variante issue de Paper 2 ne se distingue pas significativement de SMOTE classique sur ce dataset. Cela ne contredit pas l'utilité potentielle de la variante — elle est probablement utile sur des classes minoritaires beaucoup plus petites — mais montre que sur des datasets de cette taille (~3 000 minoritaires), la distribution d'interpolation est un détail.

### 7.5 Pistes d'extension

Plusieurs directions d'approfondissement émergent du projet.

**Court terme.** Optimisation des hyperparamètres XGBoost et Gamma par recherche bayésienne. Exploration d'autres variantes SMOTE (Borderline-SMOTE, ADASYN). Ajout d'une cross-validation 5-fold pour des intervalles de confiance plus robustes.

**Moyen terme.** Comparaison à des modèles plus récents : LightGBM, CatBoost, modèles d'ensemble (stacking de XGBoost et de LR calibré). Modélisation directement multi-classes (0/1/2+) avec SAMME.C2 comme dans Paper 1, en acceptant la difficulté de la classe 2+ très peu peuplée.

**Long terme.** Application des méthodes développées par Stocksieker (2024) — autoencodeurs débiaisés, GOLIATH, DAVID — pour explorer si des techniques de génération synthétique plus sophistiquées que SMOTE peuvent améliorer la performance. Adaptation à un contexte de **régression directe** sur la prime pure (modèle Tweedie) plutôt que la décomposition fréquence × sévérité.

---

## 8. Conclusion

Ce projet a exploré l'apport des données télématiques à la modélisation actuarielle automobile, en traitant en parallèle deux questions distinctes : la **segmentation du risque** par classification binaire, et la **tarification** par modélisation fréquence × sévérité.

L'apport informationnel des données télématiques est mesuré à **+0,11 d'AUC** sur la classification binaire, doublé d'un gain de **+0,08 d'AUC** par l'utilisation de modèles non-linéaires. L'écart total entre une baseline traditionnelle linéaire et le meilleur modèle non-linéaire sur données complètes est de **+0,19 d'AUC**, soit une amélioration relative de 29 %.

Sur le volet tarification, un pipeline actuariel classique Poisson × Gamma — augmenté d'une étape logistique pour gérer les claims non-payés — produit une prime pure équilibrée à 1,9 % près au niveau agrégé, avec une discrimination de 15× entre les déciles extrêmes. La calibration excellente du XGBoost baseline (ECE = 0,009) suggère qu'une approche entièrement ML est également viable pour la tarification, à condition de respecter le trade-off ranking / calibration.

Le projet identifie trois résultats méthodologiques généralisables au-delà du cas spécifique étudié :

1. **Le cost-sensitive learning domine SMOTE pour les modèles à base d'arbres** sur des problèmes de classification déséquilibrée.

2. **La calibration post-hoc (Platt scaling)** restaure intégralement la calibration d'un modèle cost-sensitive sans coût en AUC, permettant d'obtenir simultanément un bon ranker et un bon estimateur de probabilités.

3. **Les approches ML modernes et les modèles actuariels classiques sont complémentaires plutôt que substituts** : la première excelle pour le ranking et la segmentation opérationnelle, la seconde produit nativement des estimations interprétables et conformes aux exigences réglementaires de la tarification technique.

L'accessibilité des données télématiques étant aujourd'hui une réalité industrielle — par les véhicules connectés, par les applications mobiles, par les marchés de la donnée structurés — le travail présenté ici suggère que la valeur économique du passage à l'usage-based insurance n'est pleinement réalisée qu'en combinant **données enrichies** et **modèles capables de capturer les non-linéarités**. Pour les acteurs qui sauront combiner les deux, le gain potentiel en précision de tarification est substantiel.

---

## Références

Akerlof, G. A. (1970). The Market for "Lemons" : Quality Uncertainty and the Market Mechanism. *Quarterly Journal of Economics*, 84(3), 488–500.

Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). SMOTE: Synthetic Minority Over-sampling Technique. *Journal of Artificial Intelligence Research*, 16, 321–357.

Denuit, M., Maréchal, X., Pitrebois, S., & Walhin, J.-F. (2007). *Actuarial Modelling of Claim Counts: Risk Classification, Credibility and Bonus-Malus Systems*. Wiley.

Niculescu-Mizil, A., & Caruana, R. (2005). Predicting Good Probabilities with Supervised Learning. *Proceedings of the 22nd International Conference on Machine Learning*, 625–632.

So, B., Boucher, J.-P., & Valdez, E. A. (2020). Cost-sensitive Multi-class AdaBoost for Understanding Driving Behavior with Telematics. *arXiv preprint arXiv:2007.03100*.

So, B., Boucher, J.-P., & Valdez, E. A. (2021). Synthetic Dataset Generation of Driver Telematics. *arXiv preprint arXiv:2102.00252*.

Stocksieker, S. (2024). *Apport du Machine Learning dans la modélisation des valeurs rares et des données déséquilibrées : applications en assurance*. Thèse de doctorat, Université Claude Bernard Lyon 1.

Sun, Y., Kamel, M. S., Wong, A. K. C., & Wang, Y. (2007). Cost-sensitive boosting for classification of imbalanced data. *Pattern Recognition*, 40(12), 3358–3378.

Zhu, J., Zou, H., Rosset, S., & Hastie, T. (2009). Multi-class AdaBoost. *Statistics and its Interface*, 2(3), 349–360.

---

## Annexes

### A. Tableau récapitulatif des modèles et performances (test set)

| # | Modèle | Variables | AUC | G-mean (opt) | G-mean (0.5) | Calibration |
|---|--------|-----------|-----|--------------|--------------|-------------|
| 1 | LR baseline | Traditional (11) | 0,778 | 0,709 | — | n/a |
| 2 | LR full | Trad + Tel (48) | — | — | — | n/a |
| 3 | LR + cost-weighted | Trad + Tel (48) | 0,782 | 0,707 | 0,708 | dégradée |
| 4 | XGBoost baseline | Trad + Tel (48) | 0,891 | 0,804 | 0,368 | excellente |
| 5 | XGBoost + cost-weighted | Trad + Tel (48) | 0,889 | 0,808 | 0,797 | très dégradée |
| 6 | XGBoost + cost-weighted + Platt | Trad + Tel (48) | 0,889 | — | — | excellente |
| 7 | Poisson + Gamma 2-étapes | Trad + Tel (48) | 0,755 | n/a | n/a | équilibré à 1,9 % |

### B. Distribution des variables traditionnelles par cible

| Variable | Mean (claim=0) | Mean (claim=1) | Δ |
|----------|----------------|----------------|---|
| Duration | 250 | 308 | +58 |
| Age | 51,3 | 50,8 | -0,5 |
| Car_age | 5,7 | 4,8 | -0,9 |
| Credit_score | 754,8 | 731,2 | -23,6 |
| Annual_miles_drive | 12 800 | 14 600 | +1 800 |
| Years_noclaims | 16,8 | 14,2 | -2,6 |

### C. Code et reproductibilité

L'ensemble du code est organisé dans un notebook Jupyter unique (~250 cellules) couvrant les phases 0 (cleaning) à 6 (Poisson × Gamma). Toutes les graines aléatoires sont fixées à 42. Le code est exécutable sous Python 3.12 avec les dépendances suivantes :

- numpy, pandas, matplotlib, plotly
- scikit-learn ≥ 1.3
- xgboost ≥ 1.7
- statsmodels ≥ 0.14
- scipy

Le notebook complet et le dataset sont disponibles sur le repository GitHub du projet.
